**Task 2: Calculate Scale and Zero Point** 

**File:** task\_02\_scale\_zero\_point.ipynb 

**Objective** 

Calculate the quantization parameters (scale and zero point) required to map a floating-point tensor to the INT8 range. In Task 1, these were given; now you compute them yourself. 

**Formulas** 

q\_min = -128,  q\_max = 127 

scale = (x\_max - x\_min) / (q\_max - q\_min) 

zero\_point = round(q\_min - (x\_min / scale)) 

zero\_point = clip(zero\_point, q\_min, q\_max) 

**Input** 

Five separate tensors, each testing a different scenario: 

\# Tensor 1: Normal mixed range (positive and negative, includes zero) 

t1 = np.array(\[-1.5, -0.8, 0.0, 0.9, 2.3\], dtype=np.float32) 

\# Tensor 2: All positive values 

t2 = np.array(\[0.1, 0.5, 1.2, 2.0, 3.5\], dtype=np.float32) 

\# Tensor 3: All negative values 

t3 = np.array(\[-3.0, -2.1, -1.4, -0.6, -0.1\], dtype=np.float32) 

\# Tensor 4: Single unique value (constant tensor) 

t4 = np.array(\[5.0, 5.0, 5.0\], dtype=np.float32) 

\# Tensor 5: Very small floating-point values 

t5 = np.array(\[1e-9, 2e-9, -1e-9\], dtype=np.float32) 

**Implementation** 

def calculate\_scale\_zero\_point(tensor, q\_min=-128, q\_max=127): 

    # Calculate scale and zero point for affine INT8 quantization 

    # Handle edge cases: constant tensor, very small range 

    # Return: (scale, zero\_point) 

    pass 

**Edge Case Handling** 

| Edge Case | What Should Happen |
|-----------|--------------------|
| Single unique value (t4) | Must not divide by zero. Set scale to a small epsilon or 1.0. |
| Very small values (t5) | Must not lose all precision. Scale should reflect the tiny range. |
| All positive (t2) | Zero point shifts so 0.0 float maps near `q_min`. |
| All negative (t3) | Zero point shifts so 0.0 float maps near `q_max`. |

**Expected Output** 

For each of the 5 tensors, print: 

\--- Tensor: \--- 

Tensor values:        \[...\] 

Tensor min:           ? 

Tensor max:           ? 

Scale:                ? 

Zero point:           ? 

Quantized tensor:     \[...\] 

Dequantized tensor:   \[...\] 

Mean Absolute Error:  ? 

**Deliverable** 

A completed notebook showing scale, zero point, quantized tensor, dequantized tensor, and MAE for all 5 tensors with observations on each edge case.

In [1]:
import torch

In [2]:
# Tensor 1: Mixed positive and negative values
t1 = torch.tensor([-1.5, -0.8, 0.0, 0.9, 2.3], dtype=torch.float32)

# Tensor 2: All positive values
t2 = torch.tensor([0.1, 0.5, 1.2, 2.0, 3.5], dtype=torch.float32)

# Tensor 3: All negative values
t3 = torch.tensor([-3.0, -2.1, -1.4, -0.6, -0.1], dtype=torch.float32)

# Tensor 4: Constant tensor
t4 = torch.tensor([5.0, 5.0, 5.0], dtype=torch.float32)

# Tensor 5: Very small values
t5 = torch.tensor([1e-9, 2e-9, -1e-9], dtype=torch.float32)

In [3]:
def calculate_scale_zero_point(tensor, q_min=-128, q_max=127, eps=1e-12):
    """
    Calculate affine quantization parameters (scale and zero point).
    Handles constant tensors and tiny ranges.
    """

    x_min = tensor.min().item()
    x_max = tensor.max().item()

    # Edge case: constant tensor
    if abs(x_max - x_min) < eps:
        scale = 1.0
        zero_point = 0
        return scale, zero_point

    scale = (x_max - x_min) / (q_max - q_min)

    # Avoid division by zero
    scale = max(scale, eps)

    zero_point = round(q_min - (x_min / scale))
    zero_point = max(q_min, min(q_max, zero_point))

    return scale, int(zero_point)

In [4]:
def quantize_tensor(tensor, scale, zero_point, q_min=-128, q_max=127):
    q = torch.round(tensor / scale) + zero_point
    q = torch.clamp(q, q_min, q_max)
    return q.to(torch.int8)


def dequantize_tensor(q_tensor, scale, zero_point):
    return (q_tensor.to(torch.float32) - zero_point) * scale

In [5]:
tensors = {
    "Tensor 1": t1,
    "Tensor 2": t2,
    "Tensor 3": t3,
    "Tensor 4": t4,
    "Tensor 5": t5
}

for name, tensor in tensors.items():

    scale, zero_point = calculate_scale_zero_point(tensor)

    q_tensor = quantize_tensor(tensor, scale, zero_point)
    dq_tensor = dequantize_tensor(q_tensor, scale, zero_point)

    mae = torch.mean(torch.abs(tensor - dq_tensor))

    print("=" * 50)
    print(name)
    print("=" * 50)
    print("Tensor values      :", tensor.numpy())
    print("Tensor min         :", tensor.min().item())
    print("Tensor max         :", tensor.max().item())
    print("Scale              :", scale)
    print("Zero point         :", zero_point)
    print("Quantized tensor   :", q_tensor.numpy())
    print("Dequantized tensor :", dq_tensor.numpy())
    print("Mean Absolute Error:", mae.item())
    print()

Tensor 1
Tensor values      : [-1.5 -0.8  0.   0.9  2.3]
Tensor min         : -1.5
Tensor max         : 2.299999952316284
Scale              : 0.014901960597318761
Zero point         : -27
Quantized tensor   : [-128  -81  -27   33  127]
Dequantized tensor : [-1.505098   -0.80470586  0.          0.8941176   2.2949018 ]
Mean Absolute Error: 0.004156863782554865

Tensor 2
Tensor values      : [0.1 0.5 1.2 2.  3.5]
Tensor min         : 0.10000000149011612
Tensor max         : 3.5
Scale              : 0.013333333327489741
Zero point         : -128
Quantized tensor   : [-120  -90  -38   22  127]
Dequantized tensor : [0.10666667 0.50666666 1.2        2.         3.4       ]
Mean Absolute Error: 0.022666646167635918

Tensor 3
Tensor values      : [-3.  -2.1 -1.4 -0.6 -0.1]
Tensor min         : -3.0
Tensor max         : -0.10000000149011612
Scale              : 0.011372549013764251
Zero point         : 127
Quantized tensor   : [-128  -58    4   74  118]
Dequantized tensor : [-2.9        -2.10392